# Layers
Input → Embedding → LSTM → Dropout → Hidden Dense → Dropout → Output

# 1.Install Libraries

In [ ]:
!pip install datasets -q

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

from datasets import load_dataset
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from sklearn.metrics import classification_report, confusion_matrix

# 2. Load Datasets

In [ ]:
dataset=load_dataset("cardiffnlp/tweet_eval","offensive")
train_data=dataset["train"]
val_data=dataset["validation"]
test_data=dataset["test"]
print(train_data[[0]])

In [ ]:
print(dataset)
print(train_data[0:2])

# 3. Convert dataset to lists

In [ ]:
print(train_data['label'])

In [ ]:
X_train=train_data['text']
y_train=np.array(train_data['label'])

X_val=val_data['text']
y_val=np.array(val_data['label'])

X_test=test_data['text']
y_test=np.array(test_data['label'])

# 4. Tokenize text

In [ ]:
lengths=[len(text.split()) for text in X_train]
print("Max length",np.max(lengths))
print("95th percentile:", np.percentile(lengths, 95))

In [ ]:
MAX_WORDS=10000
MAX_LEN=60

tokenizer=Tokenizer(num_words=MAX_WORDS,oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq=tokenizer.texts_to_sequences(X_train)
X_val_seq=tokenizer.texts_to_sequences(X_val)
X_test_seq=tokenizer.texts_to_sequences(X_test)


# 5. Padding

In [ ]:
X_train_pad=pad_sequences(X_train_seq,maxlen=MAX_LEN,padding="post",truncating="post")
X_val_pad=pad_sequences(X_val_seq,maxlen=MAX_LEN,padding="post",truncating="post")
X_test_pad=pad_sequences(X_test_seq,maxlen=MAX_LEN,padding="post",truncating="post")


# 6. Build LSTM model

In [ ]:
model=Sequential([
    Embedding(input_dim=MAX_WORDS,output_dim=128),
    Bidirectional(LSTM(64)),
    Dropout(0.5),
    Dense(32,activation="relu"),
    Dropout(0.3),
    Dense(1,activation="sigmoid")
])
model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

model.summary()

# 7. Train model

In [ ]:
history=model.fit(X_train_pad,
                  y_train,
                  validation_data=(X_val_pad,y_val),
                  epochs=5,
                  batch_size=32,
                 )

# 8. Prediction & Error analysis

In [ ]:
y_prob=model.predict(X_test_pad).ravel()#flatten the predicted output
y_pred=(y_prob>=0.5).astype(int)
print(classification_report(y_test,y_pred, target_names=["non-offensive", "offensive"]))
print(confusion_matrix(y_test,y_pred))

The model achieves reasonable performance, but still struggles with the offensive class.

# 9. Add neutral class

In [ ]:
def predict_with_uncertainty(prob):
    if(prob>=0.6):
        return "offensive"
    elif prob<=0.4:
        return "non-offensive"
    else:
        return "neutral/uncertain"

In [ ]:
uncertain_preds=[predict_with_uncertainty(p) for p in y_prob]
result_df=pd.DataFrame({
    "text":X_test,
    "true_label":y_test,
    "toxic_probability":y_prob,
    "prediction": uncertain_preds
})
result_df.head(20)